In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64
batch_size = 128
epochs = 500 # here these are training steps, not full epochs
learning_rate = 3e-4
eval_iters = 100
dropout = 0.2
n_embd = 512
n_layer = 4
n_head = 4

cuda


In [4]:
with open('fernando_pessoa.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# train a byte level BPE tokenizer on the poems
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=50000, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"])
tokenizer.train_from_iterator([text], trainer=trainer)
tokenizer.save("fernando_pessoa_tokenizer.json")

In [5]:
def encode(text):
    return tokenizer.encode(text).ids

def decode(ids):
    return tokenizer.decode(ids)

data = torch.tensor(encode(text), dtype=torch.long)

# 80% train, 20% validation
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [6]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # mask so each token only sees the previous ones
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # attention scores
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # future positions become -inf so softmax gives them 0
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatenate the heads on the last dimension
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ transformer block: attention then feed forward """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # add & norm after each sublayer, like in the original transformer paper
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocabulary_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # small normal init (std 0.02) so training starts stable
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:] # crop to the last block_size tokens
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :] # only the last time step, (B,C)
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1) # (B,1)
            index = torch.cat((index, index_next), dim=-1) # (B, T+1)
        return index

model = GPTLanguageModel(tokenizer.get_vocab_size())
m = model.to(device)

# generate before training
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

entlyancefore grows bowl bl corners peopledreac Adonisshrine general sat 1913 ke spacealeHALAMIUink poorer daring whyoldLoveslid post poorer mute given bosom drunath LetNow glades negatrit breathingponobbedreacfe girttle wise viol hoofs Earthirts eternitiesRislls elderul arts beginunateWhoial embracespaneeakins oust smiles intreat lore fore spirit blisses wouldstHeire fanees undefined petbound five miss beautiful uncovering calm't beesame fallsightent felt hispowerckreslust purposers overto believeellJump HebeRaise Io boldercarn melancholyShe Kiss words oneutte room wanting sideLove iial beauties own place taut fringedolEPstood used hurt naturalGr leaving globedXII gle climbswest fairimming wee divinity pressing follow toys nudity Apollret erewhile sing behold ar wages herd true slept alieneathnder linkingEven olives flushedOver gall muscledmporary s willing bir begin treesEPITHALAMIUM lying 1915Gir walkmortalmoryoled pairEachSurge suff fl cloist touch talk e pairing lands mistIPTIONth

In [7]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

total_correct = 0
total_samples = 0

for iter in range(epochs):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)

    # next token accuracy on the training batch
    _, predicted = torch.max(logits, 1)
    predicted = predicted.view(yb.shape)
    correct = (predicted == yb).sum().item()
    total_samples += yb.size(0) * yb.size(1)
    total_correct += correct

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

final_accuracy = total_correct / total_samples
print(f"Final accuracy: {final_accuracy:.4f}")
print(f"train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")

step: 0, train loss: 8.2958, val loss: 8.2908
step: 100, train loss: 2.2927, val loss: 3.6108
step: 200, train loss: 0.3134, val loss: 2.4300
step: 300, train loss: 0.1097, val loss: 2.4291
step: 400, train loss: 0.0875, val loss: 2.5501
Final accuracy: 0.7902
train loss: 0.0875, val loss: 2.5501


In [11]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=1000)[0].tolist())
print(generated_chars)

 of each veined emotion,
And a red race runs through his seeing and hearing,
A great carouse of dreams seen each,
Till their importunate careering
A stopped, half-hurting point of mad joy reach.




XIV


The bridegroom aches for the end of this and lusts
To know those paps in sucking gusts,
To put his first hand on that belly's hair
And feel for the lipped lair,
The fortress made but to be taken, for which
He feels the battering ram grow large and itch.
The trembling glad bride feels all the day hot
On that still cloistered spot
Where only her nightly maiden hand did feign
A pleasure's empty gain.
And, of the others, most will whisper at this,
Knowing the spurt it is;
And children yet, that watch with looking eyes,
Will now thrill to be wise
In flesh, and with big men and women act
The liquid tickling fact
For whose taste they'll in secret corners try
They scarce know what still dry.




When with the church or feast, for these
Are outward to the day, like the green trees
 boy lay dea

In [12]:
def calculate_perplexity(loss):
    return torch.exp(loss)

# train for another 500 steps and keep track of the perplexity
def train(model, optimizer, epochs, eval_iters):
    for epoch in range(epochs):
        if epoch % eval_iters == 0:
            losses = estimate_loss()
            print(f"Epoch: {epoch}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}, perplexity: {calculate_perplexity(losses['val']):.4f}")

        xb, yb = get_batch('train')
        logits, loss = model.forward(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

train(model, optimizer, epochs, eval_iters)

Epoch: 0, train loss: 0.0783, val loss: 2.6204, perplexity: 13.7409
Epoch: 100, train loss: 0.0725, val loss: 2.6826, perplexity: 14.6224
Epoch: 200, train loss: 0.0693, val loss: 2.6309, perplexity: 13.8857
Epoch: 300, train loss: 0.0679, val loss: 2.7290, perplexity: 15.3182
Epoch: 400, train loss: 0.0662, val loss: 2.7211, perplexity: 15.1972
